# Day 8 — Train/validation/test splits and why data leakage is dangerous

Run each cell with **Shift + Enter**, from the top. We use the 150-student dataset from Days 5-7, `student_data_150.csv`: 8 real students and 142 **synthetic** ones (simulated for practice, marked in the Source column). Results depend on the data and the random seed; `random_state` makes them repeatable.

## 1. Why we need data splits

In [ ]:
# Get the dataset: Colab starts with an empty folder, so download the file the first time
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-008/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# The same 150 students as Days 5-7: 8 real ones and 142 synthetic ones
df = pd.read_csv("student_data_150.csv")

# First split: train (70%) and temp (30%); second split: temp into validation and test (15% each)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

for name, part in [("Training", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name:<10} set: {len(part):>3} rows ({len(part) / len(df):.0%})")

Training   set: 105 rows (70%)
Validation set:  22 rows (15%)
Test       set:  23 rows (15%)


In [ ]:
# Check the splits: no student appears in two of them, and together they cover everyone
train_ids, val_ids, test_ids = set(train_df["Student"]), set(val_df["Student"]), set(test_df["Student"])
print("Overlap train/validation:", len(train_ids & val_ids))
print("Overlap train/test:", len(train_ids & test_ids))
print("Overlap validation/test:", len(val_ids & test_ids))
print("All students covered:", len(train_ids | val_ids | test_ids) == len(df))

Overlap train/validation: 0
Overlap train/test: 0
Overlap validation/test: 0
All students covered: True


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Why we need unseen data: a model that can memorize its training students
features = ["StudyHours", "Attendance", "PreviousScore"]
model = DecisionTreeRegressor(random_state=42)          # no depth limit: it can memorize
model.fit(train_df[features], train_df["ExamScore"])

train_error = mean_absolute_error(train_df["ExamScore"], model.predict(train_df[features]))
test_error = mean_absolute_error(test_df["ExamScore"], model.predict(test_df[features]))
print(f"Average error on the students it trained on: {train_error:.1f} points")
print(f"Average error on new students (test set):    {test_error:.1f} points")

Average error on the students it trained on: 0.0 points
Average error on new students (test set):    11.3 points


In [ ]:
# 80/10/10: a different ratio, the same idea (more data for training, smaller evaluation sets)
train80, temp20 = train_test_split(df, test_size=0.2, random_state=42)
val10, test10 = train_test_split(temp20, test_size=0.5, random_state=42)
print("80/10/10 split:", len(train80), len(val10), len(test10))

80/10/10 split: 120 15 15


## 2. How to split data in Python

In [ ]:
# 70/15/15: separate the test set first, then split the rest into training and validation
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
validation_ratio = 0.15 / 0.85                     # 15% of all students = about 17.6% of what's left
train, validation = train_test_split(train_val, test_size=validation_ratio, random_state=42)
print("Training:", len(train))
print("Validation:", len(validation))
print("Testing:", len(test))

Training: 104
Validation: 23
Testing: 23


In [ ]:
# Check the split: is each class represented in every set?
for name, part in [("Training", train), ("Validation", validation), ("Testing", test)]:
    print(f"{name:<10} rows {len(part):>3} | " + "  ".join(f"{c} {n}" for c, n in part["Class"].value_counts().sort_index().items()))

Training   rows 104 | A 34  B 35  C 33  Unknown 2
Validation rows  23 | A 8  B 6  C 8  Unknown 1
Testing    rows  23 | A 7  B 8  C 6  Unknown 2


In [ ]:
# stratify keeps the class shares the same in every set
train_val_s, test_s = train_test_split(df, test_size=0.15, random_state=42, stratify=df["Class"])
train_s, validation_s = train_test_split(train_val_s, test_size=validation_ratio, random_state=42,
                                         stratify=train_val_s["Class"])
for name, part in [("Training", train_s), ("Validation", validation_s), ("Testing", test_s)]:
    shares = part["Class"].value_counts(normalize=True).sort_index().round(2)
    print(f"{name:<10} rows {len(part):>3} | " + "  ".join(f"{c} {s:.2f}" for c, s in shares.items()))

Training   rows 104 | A 0.33  B 0.33  C 0.32  Unknown 0.03
Validation rows  23 | A 0.30  B 0.35  C 0.30  Unknown 0.04
Testing    rows  23 | A 0.35  B 0.30  C 0.30  Unknown 0.04


In [ ]:
# ExamScore is continuous: stratify on score bands instead of the raw scores
bands = pd.cut(df["ExamScore"], bins=[0, 50, 65, 101], labels=["below 50", "50-64", "65+"])
tv_b, test_b = train_test_split(df, test_size=0.15, random_state=42, stratify=bands)
print("Score bands overall:", bands.value_counts(normalize=True).sort_index().round(2).to_dict())
print("Score bands in test:", bands[test_b.index].value_counts(normalize=True).sort_index().round(2).to_dict())
print(f"Mean ExamScore: all {df['ExamScore'].mean():.1f} | test {test_b['ExamScore'].mean():.1f}")

Score bands overall: {'below 50': 0.31, '50-64': 0.4, '65+': 0.29}
Score bands in test: {'below 50': 0.3, '50-64': 0.39, '65+': 0.3}
Mean ExamScore: all 57.4 | test 57.4


In [ ]:
# random_state: the same seed gives the same split; a different seed gives different students
for seed in [42, 7, 2026]:
    _, t = train_test_split(df, test_size=0.15, random_state=seed)
    print(f"random_state={seed:<5} test size {len(t)}, mean ExamScore {t['ExamScore'].mean():.1f}, first IDs {sorted(t['Student'])[:3]}")

random_state=42    test size 23, mean ExamScore 57.7, first IDs ['G002', 'G005', 'G011']
random_state=7     test size 23, mean ExamScore 56.1, first IDs ['G004', 'G021', 'G025']
random_state=2026  test size 23, mean ExamScore 60.0, first IDs ['G008', 'G011', 'G016']


In [ ]:
# Verify that no student is in more than one set, and save the split for the charts
ids = [set(p["Student"]) for p in (train_s, validation_s, test_s)]
print("Students in more than one set:", len((ids[0] & ids[1]) | (ids[0] & ids[2]) | (ids[1] & ids[2])))
splits = pd.concat([p.assign(Split=name) for name, p in
                    [("Training", train_s), ("Validation", validation_s), ("Test", test_s)]])
splits[["Student", "Class", "ExamScore", "Split"]].to_csv("student_splits.csv", index=False)
print("Saved student_splits.csv:", len(splits), "rows")

Students in more than one set: 0
Saved student_splits.csv: 150 rows


## 3. Evaluating model performance

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict ExamScore from three features, using the stratified split from Section 2
features = ["StudyHours", "Attendance", "Age"]
target = "ExamScore"
train, validation, test = train_s, validation_s, test_s

def evaluate(model, data):
    actual, predicted = data[target], model.predict(data[features])
    mse = mean_squared_error(actual, predicted)
    return {"MAE": mean_absolute_error(actual, predicted), "MSE": mse, "RMSE": mse ** 0.5,
            "R2": r2_score(actual, predicted)}

model = LinearRegression()
model.fit(train[features], train[target])              # learn from the training set only
print(pd.DataFrame({"Training": evaluate(model, train), "Validation": evaluate(model, validation)}).T.round(3))

              MAE      MSE    RMSE     R2
Training    8.650  118.749  10.897  0.440
Validation  6.313   70.170   8.377  0.485


In [ ]:
# What the metrics mean, by hand on five validation students
actual = validation[target].head(5).to_numpy()
predicted = model.predict(validation[features].head(5)).round(1)
errors = actual - predicted
print("Actual:   ", actual)
print("Predicted:", predicted)
print("Errors:   ", errors.round(1))
print(f"MAE  = mean(|errors|)  = {abs(errors).mean():.2f}")
print(f"MSE  = mean(errors^2)  = {(errors ** 2).mean():.2f}")
print(f"RMSE = sqrt(MSE)       = {(errors ** 2).mean() ** 0.5:.2f}")

Actual:    [39. 76. 46. 69. 47.]
Predicted: [38.1 67.9 43.2 60.1 46.9]
Errors:    [0.9 8.1 2.8 8.9 0.1]
MAE  = mean(|errors|)  = 4.16
MSE  = mean(errors^2)  = 30.70
RMSE = sqrt(MSE)       = 5.54


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor

# Compare models on the VALIDATION set; R2 compares each with always predicting the mean
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Decision tree (no limit)": DecisionTreeRegressor(random_state=42),
    "Decision tree (depth 3)": DecisionTreeRegressor(max_depth=3, random_state=42),
}
rows = {}
for name, m in models.items():
    m.fit(train[features], train[target])
    rows[name] = {"train RMSE": evaluate(m, train)["RMSE"], **{f"val {k}": v for k, v in evaluate(m, validation).items() if k != "MSE"}}
print(pd.DataFrame(rows).T.round(2))

                          train RMSE  val MAE  val RMSE  val R2
Mean baseline                  14.56     9.07     11.92   -0.04
Linear regression              10.90     6.31      8.38    0.49
Decision tree (no limit)        2.42    11.52     14.68   -0.58
Decision tree (depth 3)         9.81     7.80     11.04    0.11


In [ ]:
# Choose by validation RMSE, then evaluate the chosen model on the test set ONCE
results = {name: evaluate(m, validation)["RMSE"] for name, m in models.items()}
best = min(results, key=results.get)
print("Chosen on validation:", best)
print("Test set:", {k: round(v, 3) for k, v in evaluate(models[best], test).items()})

Chosen on validation: Linear regression
Test set: {'MAE': 9.23, 'MSE': 111.544, 'RMSE': 10.561, 'R2': 0.193}


In [ ]:
# R2 can be negative: a model worse than always predicting the mean
always_70 = DummyRegressor(strategy="constant", constant=70).fit(train[features], train[target])
print("Always predicting 70: validation R2 =", round(evaluate(always_70, validation)["R2"], 3))

Always predicting 70: validation R2 = -0.902


## 4. Overfitting and underfitting

In [ ]:
# A simple model and a complex model, compared on training and validation RMSE
def rmse(actual, predicted):
    return mean_squared_error(actual, predicted) ** 0.5     # works in every scikit-learn version

simple_model = LinearRegression().fit(train[features], train[target])
complex_model = DecisionTreeRegressor(max_depth=None, random_state=42).fit(train[features], train[target])
for name, m in [("Linear regression", simple_model), ("Decision tree (no limit)", complex_model)]:
    print(f"{name:<25} training RMSE {rmse(train[target], m.predict(train[features])):5.2f}"
          f" | validation RMSE {rmse(validation[target], m.predict(validation[features])):5.2f}")

Linear regression         training RMSE 10.90 | validation RMSE  8.38
Decision tree (no limit)  training RMSE  2.42 | validation RMSE 14.68


In [ ]:
# From too simple to too complex: tree depth against training and validation error
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(train[features], train[target])
    rows.append({"max_depth": "None" if depth is None else depth,
                 "train RMSE": rmse(train[target], tree.predict(train[features])),
                 "validation RMSE": rmse(validation[target], tree.predict(validation[features]))})
curve = pd.DataFrame(rows).round(2)
print(curve.to_string(index=False))
best = curve.loc[curve["validation RMSE"].idxmin()]
print(f"\nLowest validation RMSE: max_depth={best['max_depth']} ({best['validation RMSE']})")

max_depth  train RMSE  validation RMSE
        1       12.22            11.76
        2       11.05            10.62
        3        9.81            11.04
        4        8.66            11.53
        5        7.56            12.67
        6        6.49            12.59
        8        3.94            14.87
     None        2.42            14.68

Lowest validation RMSE: max_depth=2 (10.62)


In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validation: 5 different validation folds from training + validation, averaged
train_val_all = pd.concat([train, validation])
for name, m in [("Linear regression", LinearRegression()),
                ("Tree, max_depth=2", DecisionTreeRegressor(max_depth=2, random_state=42)),
                ("Tree, max_depth=5", DecisionTreeRegressor(max_depth=5, random_state=42)),
                ("Tree, no limit", DecisionTreeRegressor(random_state=42))]:
    scores = -cross_val_score(m, train_val_all[features], train_val_all[target], cv=5,
                              scoring="neg_root_mean_squared_error")
    print(f"{name:<18} RMSE per fold {scores.round(1)} -> mean {scores.mean():.2f}")

Linear regression  RMSE per fold [13.   9.5 13.7  9.1  8.2] -> mean 10.71
Tree, max_depth=2  RMSE per fold [15.8 11.6 15.1 11.4 10.3] -> mean 12.83
Tree, max_depth=5  RMSE per fold [15.8 11.6 14.6 13.6 12.4] -> mean 13.58
Tree, no limit     RMSE per fold [17.4 15.3 17.5 15.  13.4] -> mean 15.75


## 5. Data leakage and how to avoid it

In [ ]:
from sklearn.preprocessing import StandardScaler

# Incorrect: the scaler learns its mean and spread from ALL the data, test rows included
leaky_scaler = StandardScaler().fit(df[features])
# Correct: it learns them from the training rows only
train_scaler = StandardScaler().fit(train[features])
print("Means learned from all data:     ", leaky_scaler.mean_.round(2))
print("Means learned from training only:", train_scaler.mean_.round(2))

Means learned from all data:      [ 3.57 79.49 11.02]
Means learned from training only: [ 3.59 79.24 11.09]


In [ ]:
from sklearn.pipeline import make_pipeline

# A Pipeline fits every step on the training data, then applies the same steps to new data
pipeline = make_pipeline(StandardScaler(), LinearRegression())
pipeline.fit(train[features], train[target])
print("Pipeline, validation R2:", round(pipeline.score(validation[features], validation[target]), 4))

# The leaky version: scale everything first, then train
X_all = pd.DataFrame(leaky_scaler.transform(df[features]), index=df.index, columns=features)
leaky = LinearRegression().fit(X_all.loc[train.index], train[target])
print("Leaky scaling, validation R2:", round(leaky.score(X_all.loc[validation.index], validation[target]), 4))

Pipeline, validation R2: 0.4853
Leaky scaling, validation R2: 0.4853


In [ ]:
# Target leakage: features calculated FROM ExamScore give the answer away
leaky_data = {name: part.assign(Grade=part["ExamScore"] // 10)         # a grade band: 5 means 50-59
              for name, part in [("train", train), ("validation", validation)]}
for label, cols in [("Honest features", features),
                    ("+ Passed (score >= 50)", features + ["Passed"]),
                    ("+ Grade (score // 10)", features + ["Grade"])]:
    m = LinearRegression().fit(leaky_data["train"][cols], train[target])
    print(f"{label:<24} validation R2: {m.score(leaky_data['validation'][cols], validation[target]):.3f}")

Honest features          validation R2: 0.485
+ Passed (score >= 50)   validation R2: 0.527
+ Grade (score // 10)    validation R2: 0.951


In [ ]:
# Duplicate leakage: copies of the validation students accidentally added to the training data
train_with_copies = pd.concat([train, validation])
for name, data in [("Clean training set", train), ("Training set with copies", train_with_copies)]:
    tree = DecisionTreeRegressor(random_state=42).fit(data[features], data[target])
    print(f"{name:<25} validation RMSE {rmse(validation[target], tree.predict(validation[features])):5.2f}")

Clean training set        validation RMSE 14.68


Training set with copies  validation RMSE  0.98


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Group leakage: if each student had several records, split by student so none is in both sets
records = pd.concat([df.assign(Term=1), df.assign(Term=2)])          # pretend: two terms per student
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(splitter.split(records, groups=records["Student"]))
train_students = set(records.iloc[train_idx]["Student"])
test_students = set(records.iloc[test_idx]["Student"])
print("Records:", len(records), "| training:", len(train_idx), "| test:", len(test_idx))
print("Students in both sets:", len(train_students & test_students))

Records: 300 | training: 254 | test: 46
Students in both sets: 0


## 6. Mini project: build and evaluate your first ML model

In [ ]:
# Mini project: load the dataset, choose features and target, split 70/15/15
original = pd.read_csv("student_data_150.csv")          # kept unchanged; we work on a copy
df = original.copy()
features = ["StudyHours", "Attendance", "Age"]
target = "ExamScore"
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42)
for name, part in [("Training", train), ("Validation", validation), ("Test", test)]:
    print(f"{name:<10} {len(part):>3} students ({(part['Source'] == 'synthetic').sum()} synthetic)")

Training   104 students (98 synthetic)
Validation  23 students (21 synthetic)
Test        23 students (23 synthetic)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def evaluate(model, data):
    actual, predicted = data[target], model.predict(data[features])
    mse = mean_squared_error(actual, predicted)
    return {"MAE": mean_absolute_error(actual, predicted), "MSE": mse, "RMSE": mse ** 0.5,
            "R2": r2_score(actual, predicted)}

models = {
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Tree depth 2": DecisionTreeRegressor(max_depth=2, random_state=42),
    "Tree depth 5": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Tree depth 10": DecisionTreeRegressor(max_depth=10, random_state=42),
}
rows = []
for name, model in models.items():
    model.fit(train[features], train[target])          # every model learns from the training set only
    for set_name, data in [("Training", train), ("Validation", validation)]:
        rows.append({"Model": name, "Set": set_name, **evaluate(model, data)})
results = pd.DataFrame(rows).round(2)
print(results.to_string(index=False))

            Model        Set   MAE    MSE  RMSE    R2
Linear regression   Training  8.24 111.30 10.55  0.43
Linear regression Validation  8.83 110.24 10.50  0.31
     Tree depth 2   Training  8.96 117.14 10.82  0.40
     Tree depth 2 Validation 10.27 145.59 12.07  0.09
     Tree depth 5   Training  4.92  46.18  6.80  0.77
     Tree depth 5 Validation 11.20 188.69 13.74 -0.18
    Tree depth 10   Training  0.62   4.40  2.10  0.98
    Tree depth 10 Validation 12.65 247.26 15.72 -0.54


In [ ]:
# Choose by validation RMSE, then evaluate the chosen model on the test set once
validation_rmse = results[results["Set"] == "Validation"].set_index("Model")["RMSE"]
best = validation_rmse.idxmin()
print("Chosen on validation:", best)
print("Test set:", {k: round(v, 2) for k, v in evaluate(models[best], test).items()})

Chosen on validation: Linear regression
Test set: {'MAE': 8.62, 'MSE': 98.47, 'RMSE': 9.92, 'R2': 0.49}


In [ ]:
# Leakage checks
scaler = models["Linear regression"].named_steps["standardscaler"]
print("Scaler means = training means:", bool((scaler.mean_.round(6) == train[features].mean().to_numpy().round(6)).all()))
ids = [set(p["Student"]) for p in (train, validation, test)]
print("Students in two sets:", len((ids[0] & ids[1]) | (ids[0] & ids[2]) | (ids[1] & ids[2])))
print("Target-derived features used:", [f for f in features if f in ("ExamScore", "Passed")])
print("Original file unchanged:", original.equals(pd.read_csv("student_data_150.csv")))

Scaler means = training means: True
Students in two sets: 0
Target-derived features used: []
Original file unchanged: True


In [ ]:
# Save the results for the charts: errors per model, and the chosen model's test predictions
comparison = results.pivot(index="Model", columns="Set", values="RMSE").reset_index()
comparison.columns = ["Model", "TrainingRMSE", "ValidationRMSE"]
comparison.to_csv("model_comparison.csv", index=False)
pd.DataFrame({"ActualScore": test[target], "PredictedScore": models[best].predict(test[features]).round(1)}
             ).to_csv("test_predictions.csv", index=False)
print(comparison.to_string(index=False))

            Model  TrainingRMSE  ValidationRMSE
Linear regression         10.55           10.50
    Tree depth 10          2.10           15.72
     Tree depth 2         10.82           12.07
     Tree depth 5          6.80           13.74
